# Análisis de Negocio y Respuestas Comerciales - Dataset Refinado
## Resolución de las 8 Preguntas Clave del Negocio

Este notebook utiliza la información de los datasets crudos (`fact_ventas.json`, `fact_inventario.json`, `fact_competencia.json`, `fact_evaluacion_proveedores.json`, `fact_abastecimiento_logistica.json`) para realizar un análisis descriptivo detallado, responder preguntas comerciales complejas y generar insights estratégicos para la empresa.

---
### Estructura de Datos
Se cargan los datos directamente de los archivos JSON para mantener la máxima granularidad, incluyendo nombres, ID y fechas que son necesarias para agregaciones geográficas, temporales y por clientes.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
import warnings
warnings.filterwarnings('ignore')

# Configurar estilo premium
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['font.family'] = 'sans-serif'

# Crear carpeta para imágenes si no existe
os.makedirs('outputs_negocio', exist_ok=True)
print("Entorno inicializado correctamente.")


## 1. Cargar Datasets Crudos


In [2]:
print("Cargando archivos JSON...")
with open('fact_ventas.json', 'r', encoding='utf-8') as f:
    df_ventas = pd.DataFrame(json.load(f))
with open('fact_inventario.json', 'r', encoding='utf-8') as f:
    df_inventario = pd.DataFrame(json.load(f))
with open('fact_competencia.json', 'r', encoding='utf-8') as f:
    df_competencia = pd.DataFrame(json.load(f))
with open('fact_evaluacion_proveedores.json', 'r', encoding='utf-8') as f:
    df_evaluacion = pd.DataFrame(json.load(f))
with open('fact_abastecimiento_logistica.json', 'r', encoding='utf-8') as f:
    df_abastecimiento = pd.DataFrame(json.load(f))

print("Datasets cargados:")
print(f"  Ventas: {df_ventas.shape}")
print(f"  Inventario: {df_inventario.shape}")
print(f"  Competencia: {df_competencia.shape}")
print(f"  Evaluación Proveedores: {df_evaluacion.shape}")
print(f"  Abastecimiento Logística: {df_abastecimiento.shape}")


## 2. Extracción de Atributos Anidados (Feature Extraction)
Para realizar agrupaciones complejas, extraemos los diccionarios anidados en columnas planas de pandas.


In [3]:
# --- Procesar Ventas ---
df_ventas['mes'] = df_ventas['tiempo'].apply(lambda x: x['mes'])
df_ventas['trimestre'] = df_ventas['tiempo'].apply(lambda x: x['trimestre'])
df_ventas['anio'] = df_ventas['tiempo'].apply(lambda x: x['anio'])
df_ventas['fecha_venta'] = pd.to_datetime(df_ventas['tiempo'].apply(lambda x: x['fecha']))
df_ventas['hora_venta'] = df_ventas['tiempo'].apply(lambda x: int(x['hora'].split(':')[0]))
df_ventas['semestre'] = df_ventas['tiempo'].apply(lambda x: x['semestre'])

df_ventas['nombre_sucursal'] = df_ventas['sucursal'].apply(lambda x: x['nombre_sucursal'])
df_ventas['region_sucursal'] = df_ventas['sucursal'].apply(lambda x: x['region'])
df_ventas['categoria_producto'] = df_ventas['producto'].apply(lambda x: x['categoria'])
df_ventas['nombre_producto'] = df_ventas['producto'].apply(lambda x: x['nombre_producto'])
df_ventas['id_proveedor'] = df_ventas['producto'].apply(lambda x: x['id_proveedor'])
df_ventas['descripcion_pago'] = df_ventas['metodo_pago'].apply(lambda x: x['descripcion_pago'])
df_ventas['genero_cliente'] = df_ventas['cliente'].apply(lambda x: x['genero'])
df_ventas['fecha_nacimiento_cliente'] = pd.to_datetime(df_ventas['cliente'].apply(lambda x: x['fecha_nacimiento']))

# Calcular edad del cliente al momento de la venta
df_ventas['edad_cliente'] = df_ventas['fecha_venta'].dt.year - df_ventas['fecha_nacimiento_cliente'].dt.year

# --- Crear mapa de ID de proveedor a Nombre del Proveedor ---
prov_map = {}
for r in df_evaluacion['proveedor'].dropna():
    prov_map[r['id_proveedor']] = r['nombre_proveedor']
for r in df_abastecimiento['proveedor'].dropna():
    prov_map[r['id_proveedor']] = r['nombre_proveedor']
df_ventas['nombre_proveedor'] = df_ventas['id_proveedor'].apply(lambda x: prov_map.get(x, f"PROV_{x}"))

# --- Procesar Inventario ---
df_inventario['nombre_producto'] = df_inventario['producto'].apply(lambda x: x['nombre_producto'])
df_inventario['nombre_sucursal'] = df_inventario['sucursal'].apply(lambda x: x['nombre_sucursal'])

# --- Procesar Abastecimiento ---
df_abastecimiento['nombre_canal'] = df_abastecimiento['canal'].apply(lambda x: x['nombre_canal'])
df_abastecimiento['nombre_sucursal'] = df_abastecimiento['sucursal'].apply(lambda x: x['nombre_sucursal'])

print("Características de negocio extraídas con éxito.")


## Pregunta 1: Desempeño Financiero de Sucursales en Q4 (Último Trimestre)
*¿Qué sucursales generan los mayores niveles de ingresos totales, utilidad neta y margen de rentabilidad durante el último trimestre del año, considerando las ventas registradas por producto y método de pago?*

Analizamos la rentabilidad del periodo crítico del año agrupando por sucursal, producto y método de pago en el Trimestre 4.


In [4]:
df_q4 = df_ventas[df_ventas['trimestre'] == 4]

# Resumen general por sucursal
suc_q4_summary = df_q4.groupby('nombre_sucursal').agg(
    ingresos_totales=('monto_total', 'sum'),
    costo_total=('costo_total', 'sum'),
    utilidad_neta=('utilidad_bruta', 'sum')
).reset_index()
suc_q4_summary['margen_rentabilidad'] = (suc_q4_summary['utilidad_neta'] / suc_q4_summary['ingresos_totales']).round(4)
suc_q4_summary = suc_q4_summary.sort_values(by='ingresos_totales', ascending=False)

print("--- TOP SUCURSALES EN INGRESOS Y UTILIDAD EN Q4 ---")
print(suc_q4_summary.to_string(index=False))

# Guardar desglose por producto y método de pago
suc_prod_pago_q4 = df_q4.groupby(['nombre_sucursal', 'categoria_producto', 'descripcion_pago']).agg(
    ingresos_totales=('monto_total', 'sum'),
    utilidad_neta=('utilidad_bruta', 'sum'),
    transacciones=('monto_total', 'count')
).reset_index()
suc_prod_pago_q4['margen_rentabilidad'] = (suc_prod_pago_q4['utilidad_neta'] / suc_prod_pago_q4['ingresos_totales']).round(4)

# Visualización
plt.figure(figsize=(14, 6))
sns.barplot(data=suc_q4_summary.head(10), x='ingresos_totales', y='nombre_sucursal', hue='margen_rentabilidad', palette='Reds', dodge=False)
plt.title('Ingresos Totales y Margen de Rentabilidad por Sucursal (Q4)', fontsize=14, fontweight='bold')
plt.xlabel('Ingresos Totales ($)')
plt.ylabel('Sucursal')
plt.tight_layout()
plt.savefig('outputs_negocio/p1_sucursales_q4.png', dpi=300)
plt.show()


## Pregunta 2: Desempeño Financiero en Q4 por Rango de Edad
*¿Qué sucursales generan los mayores niveles de ingresos totales, utilidad neta y margen de rentabilidad durante el último trimestre del año, considerando las ventas registradas por producto y método de pago por rango de edad?*

Segmentamos a los clientes por edad para identificar qué grupos demográficos generan las mayores ganancias en cada sucursal.


In [5]:
def get_age_range(age):
    if age < 25: return '18-24 (Jóvenes)'
    elif age <= 40: return '25-40 (Adultos Jóvenes)'
    elif age <= 60: return '41-60 (Adultos)'
    else: return '60+ (Adultos Mayores)'

df_ventas['rango_edad'] = df_ventas['edad_cliente'].apply(get_age_range)
df_q4_age = df_ventas[df_ventas['trimestre'] == 4]

suc_age_q4 = df_q4_age.groupby(['nombre_sucursal', 'rango_edad']).agg(
    ingresos_totales=('monto_total', 'sum'),
    utilidad_neta=('utilidad_bruta', 'sum'),
    transacciones=('monto_total', 'count')
).reset_index()
suc_age_q4['margen_rentabilidad'] = (suc_age_q4['utilidad_neta'] / suc_age_q4['ingresos_totales']).round(4)

# Mostrar Top 10 combinaciones Sucursal - Rango de Edad
suc_age_top = suc_age_q4.sort_values(by='ingresos_totales', ascending=False).head(10)
print("--- TOP 10 SEGMENTOS SUCURSAL - RANGO EDAD EN Q4 ---")
print(suc_age_top.to_string(index=False))

# Visualización
plt.figure(figsize=(14, 6))
sns.barplot(data=suc_age_top, x='ingresos_totales', y='nombre_sucursal', hue='rango_edad', palette='Blues')
plt.title('Top Segmentos por Rango de Edad y Sucursal (Q4 - Ingresos)', fontsize=14, fontweight='bold')
plt.xlabel('Ingresos Totales ($)')
plt.ylabel('Sucursal')
plt.tight_layout()
plt.savefig('outputs_negocio/p2_sucursales_edad_q4.png', dpi=300)
plt.show()


## Pregunta 3: Impacto de Descuentos en Ventas y Rentabilidad
*¿Cómo impactan los diferentes niveles de descuento aplicados en las promociones sobre el volumen de ventas, los ingresos generados y el margen de rentabilidad, analizados por categoría de producto, región en periodos de tiempo?*

Evaluamos si los descuentos realmente incrementan la ganancia neta o si erosionan demasiado los márgenes.


In [6]:
def get_discount_range(pct):
    if pct == 0: return '0% (Sin Descuento)'
    elif pct <= 0.10: return '1-10% (Bajo)'
    elif pct <= 0.20: return '11-20% (Medio)'
    else: return '>20% (Alto)'

df_ventas['rango_descuento'] = df_ventas['descuento_pct'].apply(get_discount_range)
df_ventas['periodo'] = df_ventas['anio'].astype(str) + "-S" + df_ventas['semestre'].astype(str)

# Resumen general de descuentos
desc_summary = df_ventas.groupby('rango_descuento').agg(
    transacciones=('monto_total', 'count'),
    cantidad_vendida=('cantidad', 'sum'),
    ingresos_totales=('monto_total', 'sum'),
    utilidad_bruta=('utilidad_bruta', 'sum')
).reset_index()
desc_summary['margen_promedio'] = (desc_summary['utilidad_bruta'] / desc_summary['ingresos_totales']).round(4)
print("--- RESUMEN DE IMPACTO DE DESCUENTOS ---")
print(desc_summary.to_string(index=False))

# Desglose por categoría, región y periodo
desc_detailed = df_ventas.groupby(['rango_descuento', 'categoria_producto', 'region_sucursal', 'periodo']).agg(
    volumen_ventas=('cantidad', 'sum'),
    ingresos_generados=('monto_total', 'sum'),
    utilidad_neta=('utilidad_bruta', 'sum')
).reset_index()
desc_detailed['margen_rentabilidad'] = (desc_detailed['utilidad_neta'] / desc_detailed['ingresos_generados']).round(4)

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(data=desc_summary, x='rango_descuento', y='cantidad_vendida', ax=axes[0], palette='crest')
axes[0].set_title('Volumen Vendido por Nivel de Descuento')
axes[0].set_xlabel('Rango de Descuento')
axes[0].set_ylabel('Cantidad de Productos Vendidos')

sns.barplot(data=desc_summary, x='rango_descuento', y='margen_promedio', ax=axes[1], palette='flare')
axes[1].set_title('Margen de Rentabilidad por Nivel de Descuento')
axes[1].set_xlabel('Rango de Descuento')
axes[1].set_ylabel('Margen Financiero')

plt.suptitle('Efecto de Descuentos sobre Volumen y Margen de Rentabilidad', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs_negocio/p3_impacto_descuentos.png', dpi=300)
plt.show()


## Pregunta 4: Demanda Horaria vs Niveles de Stock (Redistribución de Inventario)
*¿Qué productos presentan mayor demanda por sucursal y horario, y cómo se relaciona esta demanda con los niveles de stock para identificar si es necesario redistribuir inventario entre sucursales?*

Analizamos la demanda de productos por horario comercial y la comparamos con el stock actual disponible en cada sucursal.


In [7]:
def get_hour_range(h):
    if h < 12: return 'Mañana (08:00 - 12:00)'
    elif h < 18: return 'Tarde (12:00 - 18:00)'
    else: return 'Noche (18:00 - 22:00)'

df_ventas['rango_horario'] = df_ventas['hora_venta'].apply(get_hour_range)

# Demanda horaria por sucursal
demand_horaria = df_ventas.groupby(['nombre_producto', 'nombre_sucursal', 'rango_horario']).agg(
    cantidad_vendida=('cantidad', 'sum')
).reset_index()

# Stock actual (último registro registrado en inventario para cada producto y sucursal)
df_inv_latest = df_inventario.sort_values(by='id_tiempo').groupby(['nombre_producto', 'nombre_sucursal']).last().reset_index()

# Unir demanda y stock
stock_vs_demand = pd.merge(
    demand_horaria.groupby(['nombre_producto', 'nombre_sucursal']).agg(demanda_total=('cantidad_vendida', 'sum')).reset_index(),
    df_inv_latest[['nombre_producto', 'nombre_sucursal', 'stock_actual', 'stock_minimo']],
    on=['nombre_producto', 'nombre_sucursal'],
    how='inner'
)

# Identificar sobre-stock y quiebre de stock potencial
stock_vs_demand['alerta_quiebre'] = stock_vs_demand['stock_actual'] < stock_vs_demand['stock_minimo']
stock_vs_demand['alerta_exceso'] = stock_vs_demand['stock_actual'] > (stock_vs_demand['stock_minimo'] * 3)

# Identificar productos que necesitan redistribución
print("--- PRODUCTOS EN RIESGO DE QUIEBRE (DEMANDA > STOCK) ---")
print(stock_vs_demand[stock_vs_demand['alerta_quiebre']].sort_values(by='demanda_total', ascending=False).head(10)[['nombre_producto', 'nombre_sucursal', 'demanda_total', 'stock_actual', 'stock_minimo']])

print("\n--- PRODUCTOS CON EXCESO DE STOCK ---")
print(stock_vs_demand[stock_vs_demand['alerta_exceso']].sort_values(by='stock_actual', ascending=False).head(10)[['nombre_producto', 'nombre_sucursal', 'demanda_total', 'stock_actual', 'stock_minimo']])


## Pregunta 5: Sensibilidad al Precio de la Competencia
*¿Qué productos muestran una reducción significativa en el volumen de ventas o número de transacciones cuando el precio del competidor es inferior al precio ofrecido por la empresa en determinados periodos de tiempo o regiones?*

Identificamos los productos más sensibles al precio uniendo las ventas con el monitoreo de competencia.


In [8]:
# Unir ventas con competencia por producto, tiempo y región
df_comp_sales = pd.merge(
    df_ventas,
    df_competencia[['id_producto', 'id_tiempo', 'precio_competidor', 'competidor']],
    on=['id_producto', 'id_tiempo'],
    how='inner'
)

df_comp_sales['competidor_barato'] = df_comp_sales['precio_competidor'] < df_comp_sales['precio_unitario']

# Analizar volumen de ventas promedio cuando el competidor es más barato vs cuando no
sensibilidad = df_comp_sales.groupby(['nombre_producto', 'competidor_barato']).agg(
    volumen_ventas=('cantidad', 'sum'),
    transacciones=('cantidad', 'count')
).reset_index()

# Pivotar para comparar
sens_pivot = sensibilidad.pivot(index='nombre_producto', columns='competidor_barato', values='volumen_ventas').fillna(0)
sens_pivot.columns = ['Vol_Competidor_Caro_Igual', 'Vol_Competidor_Barato']
sens_pivot['reduccion_volumen'] = sens_pivot['Vol_Competidor_Caro_Igual'] - sens_pivot['Vol_Competidor_Barato']
sens_pivot['pct_reduccion'] = (sens_pivot['reduccion_volumen'] / sens_pivot['Vol_Competidor_Caro_Igual']).round(4) * 100

print("--- PRODUCTOS MÁS SENSIBLES AL PRECIO DE LA COMPETENCIA (MAYOR CAÍDA DE VENTAS) ---")
print(sens_pivot[sens_pivot['reduccion_volumen'] > 0].sort_values(by='pct_reduccion', ascending=False).head(10))


## Pregunta 6: Categorías Inelásticas / Estables ante la Competencia
*¿Qué categorías de productos mantienen niveles estables de ventas o transacciones incluso cuando los precios del competidor son menores durante determinados periodos o regiones?*

Identificamos las categorías donde el cliente mantiene lealtad a la empresa a pesar de que el competidor tenga menor precio.


In [9]:
sens_cat = df_comp_sales.groupby(['categoria_producto', 'competidor_barato']).agg(
    volumen_ventas=('cantidad', 'sum'),
    transacciones=('cantidad', 'count')
).reset_index()

sens_cat_pivot = sens_cat.pivot(index='categoria_producto', columns='competidor_barato', values='volumen_ventas').fillna(0)
sens_cat_pivot.columns = ['Vol_Competidor_Caro_Igual', 'Vol_Competidor_Barato']
sens_cat_pivot['diferencia'] = sens_cat_pivot['Vol_Competidor_Barato'] - sens_cat_pivot['Vol_Competidor_Caro_Igual']
sens_cat_pivot['pct_cambio'] = (sens_cat_pivot['diferencia'] / sens_cat_pivot['Vol_Competidor_Caro_Igual']).round(4) * 100

# Ordenar por el cambio porcentual de mayor estabilidad (menor caída o incremento positivo)
print("--- ESTABILIDAD DE VENTAS POR CATEGORÍA DE PRODUCTO ---")
print(sens_cat_pivot.sort_values(by='pct_cambio', ascending=False))

# Visualización
plt.figure(figsize=(12, 6))
sns.barplot(data=sens_cat, x='categoria_producto', y='volumen_ventas', hue='competidor_barato', palette='Set2')
plt.title('Comparativa de Ventas por Categoría según Precio del Competidor', fontsize=14, fontweight='bold')
plt.xlabel('Categoría de Producto')
plt.ylabel('Cantidad Vendida')
plt.xticks(rotation=45)
plt.legend(title='Competidor es más barato')
plt.tight_layout()
plt.savefig('outputs_negocio/p6_categorias_lealtad.png', dpi=300)
plt.show()


## Pregunta 7: Desempeño y Volumen por Proveedor
*¿Qué proveedores abastecen los productos que generan el mayor volumen de ventas e ingresos en las diferentes sucursales, y cómo varía su desempeño según la categoría de producto y el periodo de tiempo?*

Evaluamos la participación de cada proveedor en los ingresos y utilidad de la empresa.


In [10]:
prov_perf = df_ventas.groupby(['nombre_proveedor', 'nombre_sucursal', 'categoria_producto', 'periodo']).agg(
    volumen_ventas=('cantidad', 'sum'),
    ingresos_totales=('monto_total', 'sum'),
    utilidad_neta=('utilidad_bruta', 'sum')
).reset_index()
prov_perf['margen_rentabilidad'] = (prov_perf['utilidad_neta'] / prov_perf['ingresos_totales']).round(4)

# Resumen general por Proveedor
prov_summary = df_ventas.groupby('nombre_proveedor').agg(
    volumen_ventas=('cantidad', 'sum'),
    ingresos_totales=('monto_total', 'sum'),
    utilidad_neta=('utilidad_bruta', 'sum')
).reset_index().sort_values(by='ingresos_totales', ascending=False)

print("--- PARTICIPACIÓN FINANCIERA GENERAL POR PROVEEDOR ---")
print(prov_summary.to_string(index=False))

# Visualización Top 5 Proveedores
plt.figure(figsize=(10, 6))
sns.barplot(data=prov_summary.head(5), x='ingresos_totales', y='nombre_proveedor', palette='viridis')
plt.title('Ingresos Totales Generados por los Top 5 Proveedores', fontsize=14, fontweight='bold')
plt.xlabel('Ingresos Totales ($)')
plt.ylabel('Proveedor')
plt.tight_layout()
plt.savefig('outputs_negocio/p7_proveedores.png', dpi=300)
plt.show()


## Pregunta 8: Eficiencia de Canales de Distribución y Abastecimiento
*¿Qué canales de distribución permiten abastecer con mayor eficiencia a las sucursales que presentan mayor demanda de productos, considerando tiempos de reposición, volumen de ventas y disponibilidad de inventario?*

Analizamos la logística de abastecimiento cruzando los tiempos de entrega, volumen y costos por canal de distribución.


In [11]:
df_abastecimiento['eficiencia_entrega_pct'] = (df_abastecimiento['cantidad_recibida'] / df_abastecimiento['cantidad_solicitada']).round(4) * 100

# Agrupación por Canal de Distribución
canal_perf = df_abastecimiento.groupby('nombre_canal').agg(
    ordenes_totales=('id_logistica', 'count'),
    cantidad_solicitada=('cantidad_solicitada', 'sum'),
    cantidad_recibida=('cantidad_recibida', 'sum'),
    tiempo_entrega_promedio=('tiempo_entrega_dias', 'mean'),
    costo_logistico_total=('costo_logistico', 'sum')
).reset_index()
canal_perf['eficiencia_cumplimiento'] = (canal_perf['cantidad_recibida'] / canal_perf['cantidad_solicitada']).round(4) * 100
canal_perf['costo_logistico_por_unidad'] = (canal_perf['costo_logistico_total'] / canal_perf['cantidad_recibida']).round(4)

print("--- DESEMPEÑO Y EFICIENCIA DE LOS CANALES DE DISTRIBUCIÓN ---")
print(canal_perf.to_string(index=False))

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(data=canal_perf, x='nombre_canal', y='tiempo_entrega_promedio', ax=axes[0], palette='Blues_r')
axes[0].set_title('Tiempo Promedio de Reposición (Días)')
axes[0].set_xlabel('Canal de Distribución')
axes[0].set_ylabel('Días')

sns.barplot(data=canal_perf, x='nombre_canal', y='eficiencia_cumplimiento', ax=axes[1], palette='Greens_r')
axes[1].set_title('Tasa de Cumplimiento de Órdenes (%)')
axes[1].set_xlabel('Canal de Distribución')
axes[1].set_ylabel('% Cumplimiento')

plt.suptitle('Eficiencia Logística de Canales de Abastecimiento', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs_negocio/p8_eficiencia_logistica.png', dpi=300)
plt.show()


## 3. Exportar Resultados Consolidados a Excel
Exportamos todos los análisis y tablas de resultados generadas a un libro de Excel consolidado `respuestas_negocio.xlsx` para fácil consulta y reportes corporativos.


In [12]:
print("Guardando datos consolidados en Excel...")
with pd.ExcelWriter('respuestas_negocio.xlsx') as writer:
    suc_q4_summary.to_excel(writer, sheet_name='Q1_Rentabilidad_Sucursales', index=False)
    suc_prod_pago_q4.to_excel(writer, sheet_name='Q1_Desglose_Ventas_Q4', index=False)
    suc_age_q4.to_excel(writer, sheet_name='Q2_Rentabilidad_por_Edad', index=False)
    desc_summary.to_excel(writer, sheet_name='Q3_Resumen_Descuentos', index=False)
    desc_detailed.to_excel(writer, sheet_name='Q3_Detalle_Descuentos', index=False)
    stock_vs_demand.to_excel(writer, sheet_name='Q4_Demanda_vs_Stock', index=False)
    sens_pivot.to_excel(writer, sheet_name='Q5_Sensibilidad_Competencia', index=False)
    sens_cat_pivot.to_excel(writer, sheet_name='Q6_Estabilidad_Categorias', index=False)
    prov_summary.to_excel(writer, sheet_name='Q7_Participacion_Proveedores', index=False)
    prov_perf.to_excel(writer, sheet_name='Q7_Desempeño_Detallado_Prov', index=False)
    canal_perf.to_excel(writer, sheet_name='Q8_Eficiencia_Canales', index=False)

print("¡Excel respuestas_negocio.xlsx guardado con éxito!")
